# Más allá: la banda de dependencia con cada agregado y un veredicto honesto de replicación

- **Nivel:** 4, Más allá del estado del arte
- **Requisitos previos:** los cuadernos 01 a 03 y 08 de este tema (la escalera, la banda, las cotas y el estado del arte)
- **Tiempo de lectura:** unos 45 minutos

## Objetivos

Al terminar este cuaderno puedes:

- enunciar con precisión la afirmación BS-9 del producto (toda cifra de capital agregado de C18 se muestra con la banda de dependencia, nunca como una sola cifra diversificada) y cuál es su evidencia;
- comprobar la afirmación en cada documento versionado de C18 que lleva un agregado: ocho filas de banda en un orden fijo, una fila rotulada capital principal y las otras información, las filas en forma cerrada recalculadas con el motor, las reglas de intervalo y de signo, y la asignación de Euler sumando;
- ver al motor devolver la **banda completa** sobre un ejemplo con valores de referencia, en vez de un solo número;
- decir qué ocultaría una sola cifra, medido sobre la banda versionada;
- leer el registro de preparación del caso y decir con honestidad qué caso del tema está planificado y no construido, y por qué.

## Preparación

Instala una vez el entorno fijado, en un entorno virtual, con el comando de siempre:

```bash
pip install -r notebooks/requirements.txt
```

o solo los paquetes que usa este cuaderno, con las mismas versiones exactas:

```bash
pip install riskvalidation==0.8.1 numpy==2.4.6 scipy==1.18.1
```

`scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` en Windows) construye `.venv-notebooks` por ti. Un cuaderno nunca instala nada por sí mismo. Los identificadores de las pruebas del motor se imprimen tal como los escribe el motor. Sus mensajes y notas están escritos en inglés: el árbol inglés los imprime como están, y este árbol imprime una traducción desde una tabla del cuaderno que falla si el motor cambia el texto.

La celda siguiente importa lo que usa el cuaderno e imprime las versiones de los paquetes, para que las compares con las fijadas.

In [1]:
# Importar el motor y las ayudas, e imprimir las versiones de los paquetes que usa este cuaderno
import hashlib
import importlib.metadata as md
import json
from pathlib import Path

import numpy as np
import riskvalidation as rv
from riskvalidation.aggregation import band as rv_band
from riskvalidation.aggregation import copulas as cop
from riskvalidation.aggregation import simple
from riskvalidation.validation import aggregation as agg_tests
from scipy import stats
for name in ("riskvalidation", "numpy", "scipy",):
    print(name, md.version(name))

riskvalidation 0.8.1
numpy 2.4.6
scipy 1.18.1


## Datos

Este cuaderno lee cada agregado que **C18** versiona, y los registros que dicen qué está y qué no está construido.

- **Propietario y licencia:** los documentos los escribe el pipeline sin conexión de este repositorio (`data-pipeline/pipeline/cases/c18/`) con el motor `riskvalidation` y se versionan en `data/derived/C18/`. Las trayectorias de escenario vienen de la Reserva Federal (dominio público), de la EBA y la ESRB (reproducción autorizada citando la fuente) y de la NGFS (solo derivado: únicamente las pocas variables que usa el caso). **Banco Andino es un banco sintético**: su cartera y su capital están calibrados a agregados públicos, y las trayectorias son trayectorias reales de Estados Unidos y de la zona euro trasladadas a su punto de partida, así que los satélites son una adaptación calibrada en EE. UU.
- **Estatus de verdad:** `synthetic-calibrated`. No hay respuesta publicada para el banco; las verificaciones de este cuaderno son recálculos de números versionados con el motor, no comparaciones con un resultado publicado.
- **Horneado con:** `riskvalidation` 0.8.1 (semilla 42), la versión que instala este cuaderno, así que se espera que un recálculo coincida hasta el redondeo.
- **Archivos leídos:** los cuatro documentos de escenario `data/derived/C18/baseline.json`, `eba-2025-adverse.json`, `fed-2026-severely-adverse.json` y `ngfs-delayed-transition.json` y el documento de agregación `t-copula-baseline.json` (cada uno lleva `outputs.aggregation`, y el último también el estrés mercado-crédito), el índice de casos `data/derived/manifests/index.json`, el registro de preparación `data/derived/readiness.json` y el contrato de estrés del motor `docs/contracts/engine/stress-api.md` (su párrafo de alcance registra por qué el caso C19 sigue planificado).
- **Qué se afirma y qué no.** La afirmación es sobre lo que **hace Contraste**: muestra la banda. No es una afirmación de que nadie más lo hace, lo que la investigación no pudo establecer más allá de su propia búsqueda.

In [2]:
# El cuaderno está tres carpetas bajo la raíz del repositorio; un archivo ausente detiene el cuaderno y nombra su ruta
ROOT = Path("../../..").resolve()
DERIVED = ROOT / "data" / "derived"


def read_json(relative):
    return json.loads((DERIVED / relative).read_text(encoding="utf-8"))


def sino(valor):
    # Mostrar un valor verdadero o falso como "sí" o "no"
    return "sí" if valor else "no"


print("carpeta derived encontrada:", sino(DERIVED.is_dir()))

SCENARIOS = ("baseline", "eba-2025-adverse", "fed-2026-severely-adverse", "ngfs-delayed-transition")
blocks = {name: read_json(f"C18/{name}.json")["outputs"]["aggregation"] for name in SCENARIOS}
second = read_json("C18/t-copula-baseline.json")["outputs"]
blocks["t-copula-baseline"] = second["aggregation"]
blocks["t-copula-baseline, mercado-crédito 0.8"] = second["correlation_stress"]["aggregation"]
print("documentos con un agregado:", len(blocks))
for name in blocks:
    print(" -", name)

carpeta derived encontrada: sí
documentos con un agregado: 6
 - baseline
 - eba-2025-adverse
 - fed-2026-severely-adverse
 - ngfs-delayed-transition
 - t-copula-baseline
 - t-copula-baseline, mercado-crédito 0.8


## Método

### Paso 1: la afirmación, comprobada en cada agregado

**La afirmación (BS-9).** Toda cifra de capital agregado de C18 se muestra con la **banda** de métodos de agregación sobre las mismas marginales, un horizonte y un nivel de confianza: la cota inferior, la suma simple, la regla de la raíz cuadrada, la cópula gaussiana, una cópula t por cada grado de libertad del conjunto nombrado y el peor caso exacto por el algoritmo de reordenamiento, en ese orden. La suma simple es la única fila rotulada **capital principal**; toda otra fila, la asignación de Euler y las medidas de riesgo de modelo se rotulan **información, no capital**. El caso nunca muestra una sola cifra diversificada.

La celda siguiente comprueba la afirmación de forma estructural en los cinco documentos (seis agregados, contando el estrés de correlación, de los cuales solo dos son distintos, lo que la celda muestra con un hash): el número y el orden de las filas, el estado de cada una, que exactamente una fila es el capital principal y es la suma simple, que las contribuciones de Euler suman y que el bloque de riesgo de modelo está rotulado como información. Imprime una línea por documento.

In [3]:
# Comprobar la estructura de la banda en cada agregado versionado: filas, orden, estado, capital principal, aditividad de Euler, rótulo de riesgo de modelo
EXPECTED = ["lower-bound", "simple-sum", "variance-covariance", "gaussian-copula", "t-copula", "t-copula", "t-copula", "worst-case"]
for name, block in blocks.items():
    methods = [row["method"] for row in block["band"]]
    headline_rows = [row["method"] for row in block["band"] if row["status"] == "headline-capital"]
    others = {row["status"] for row in block["band"] if row["method"] != "simple-sum"}
    worst = next(row["capital_clp_m"] for row in block["band"] if row["method"] == "worst-case")
    print(f"{name:38s} filas {len(methods)} | orden como se dijo: {sino(methods == EXPECTED)} | filas principales: {headline_rows} | las otras: {sorted(others)}")
    print(f"{'':38s} principal {block['headline']['capital_clp_m']:,.0f} | peor caso sobre principal {worst / block['headline']['capital_clp_m']:.4f} | brecha de Euler {block['euler']['additivity']['relative_error']:.1e} | riesgo de modelo: {block['model_risk']['status']} | un nivel y un horizonte: {block['confidence_level']}, {block['horizon_years']}")
# La agregación no depende del escenario: mostrar cuáles de los seis bloques son un mismo bloque
digest = {name: hashlib.sha256(json.dumps(block, sort_keys=True).encode("utf-8")).hexdigest() for name, block in blocks.items()}
stressed_name = "t-copula-baseline, mercado-crédito 0.8"
same_as_baseline = [name for name in blocks if digest[name] == digest["baseline"]]
print("bloques idénticos al del base:", same_as_baseline)
print("agregados distintos entre los seis:", len(set(digest.values())), "| sha256 del bloque base:", digest["baseline"][:12], "| del bloque con estrés:", digest[stressed_name][:12])

baseline                               filas 8 | orden como se dijo: sí | filas principales: ['simple-sum'] | las otras: ['information-not-capital']
                                       principal 658,046 | peor caso sobre principal 1.2512 | brecha de Euler 0.0e+00 | riesgo de modelo: information-not-capital | un nivel y un horizonte: 0.999, 1
eba-2025-adverse                       filas 8 | orden como se dijo: sí | filas principales: ['simple-sum'] | las otras: ['information-not-capital']
                                       principal 658,046 | peor caso sobre principal 1.2512 | brecha de Euler 0.0e+00 | riesgo de modelo: information-not-capital | un nivel y un horizonte: 0.999, 1
fed-2026-severely-adverse              filas 8 | orden como se dijo: sí | filas principales: ['simple-sum'] | las otras: ['information-not-capital']
                                       principal 658,046 | peor caso sobre principal 1.2512 | brecha de Euler 0.0e+00 | riesgo de modelo: information-not-cap

### Paso 2: recalcular lo que se puede recalcular, en cada agregado

La estructura no basta: las filas deben ser correctas. Dos filas de la banda dependen solo de los capitales de las marginales y de la matriz de correlación, así que el motor las recalcula exactamente desde cada documento: la suma simple y la regla de la raíz cuadrada. Las reglas que atan el resto son las pruebas del motor: `worst_case_bracket` (toda fila dentro del intervalo) y `diversification_sign` (ninguna fila diversificada sobre la suma simple), y el documento con estrés debe tener una fila de raíz cuadrada mayor que el base, porque una correlación más alta no puede bajar el capital. La celda siguiente corre todo eso sobre los seis agregados.

In [4]:
# Recalcular la suma simple y la fila de raíz cuadrada de cada agregado y correr sobre él las pruebas de intervalo y de signo
for name, block in blocks.items():
    ec = np.array([m["ec_clp_m"] for m in block["marginals"]])
    corr = np.array(block["correlation"]["matrix"])
    rows = {(r["method"], r["nu"]): r["capital_clp_m"] for r in block["band"]}
    exact = sino(np.isclose(simple.simple_sum(ec), rows[("simple-sum", None)]) and np.isclose(simple.variance_covariance(ec, corr), rows[("variance-covariance", None)]))
    between = [v for (method, nu), v in rows.items() if method not in ("lower-bound", "worst-case")]
    bracket = agg_tests.worst_case_bracket(rows[("lower-bound", None)], between, rows[("worst-case", None)])
    factors = [r["diversification_factor"] for r in block["band"] if r["method"] in ("variance-covariance", "gaussian-copula", "t-copula")]
    sign = agg_tests.diversification_sign(factors)
    print(f"{name:38s} filas cerradas recalculadas exactamente: {exact} | intervalo: {bracket.light.value} | signo: {sign.light.value}")
base_vc = next(r["capital_clp_m"] for r in blocks["t-copula-baseline"]["band"] if r["method"] == "variance-covariance")
stress_vc = next(r["capital_clp_m"] for r in blocks["t-copula-baseline, mercado-crédito 0.8"]["band"] if r["method"] == "variance-covariance")
print("fila de raíz cuadrada, base y con estrés:", round(base_vc, 1), round(stress_vc, 1), "| una correlación más alta no la baja:", sino(stress_vc >= base_vc))

baseline                               filas cerradas recalculadas exactamente: sí | intervalo: green | signo: green
eba-2025-adverse                       filas cerradas recalculadas exactamente: sí | intervalo: green | signo: green
fed-2026-severely-adverse              filas cerradas recalculadas exactamente: sí | intervalo: green | signo: green
ngfs-delayed-transition                filas cerradas recalculadas exactamente: sí | intervalo: green | signo: green
t-copula-baseline                      filas cerradas recalculadas exactamente: sí | intervalo: green | signo: green
t-copula-baseline, mercado-crédito 0.8 filas cerradas recalculadas exactamente: sí | intervalo: green | signo: green
fila de raíz cuadrada, base y con estrés: 528106.4 568520.5 | una correlación más alta no la baja: sí


### Paso 3: el motor devuelve la banda completa

La afirmación descansa en el motor: `dependence_band` devuelve una `Band` de filas y un capital principal, nunca un escalar, y su docstring da un ejemplo con valores de referencia: capitales 3 y 4, pérdidas esperadas 0, correlación 0,5, que deben dar una fila de suma simple de 7,0, una fila de raíz cuadrada de 6,0828 y una cota inferior a lo más igual a la fila de raíz cuadrada, que es a lo más el peor caso. La celda siguiente construye dos marginales normales con esos capitales al 99,9 por ciento, corre la banda a un tamaño reducido (200.000 simulaciones, 20.000 celdas de cota), imprime las filas con su estado y comprueba los valores de referencia y el orden. Imprime también la línea del docstring que registra el ejemplo publicado de las cotas de peor caso, que el cuaderno 03 reprodujo.

In [5]:
# El ejemplo con valores de referencia del motor: dos marginales normales con capitales 3 y 4, y toda la banda que devuelve
import inspect
import re

from riskvalidation.aggregation import bounds

z = stats.norm.ppf(0.999)
grid = stats.norm.ppf((np.arange(1, 100_001) - 0.5) / 100_000)
golden = rv_band.dependence_band([cop.marginal_ppf_sample(3 / z * grid), cop.marginal_ppf_sample(4 / z * grid)], np.array([3.0, 4.0]), np.array([0.0, 0.0]),
                                 np.array([[1.0, 0.5], [0.5, 1.0]]), [4.0, 30.0], level=0.999, horizon_years=1, n_draws=200_000, seed=1,
                                 chunk_size=100_000, bounds_n=20_000, bounds_epsilon=1e-3, ec_consistency_tolerance=0.05)
print("objeto devuelto:", type(golden).__name__, "| capital principal:", golden.headline, "| filas:", len(golden.rows))
for row in golden.rows:
    print(f"{row.method:20s} nu {str(row.nu):5s} capital {row.capital:8.4f}  parte de la suma {row.diversification_factor:.4f}  {row.status}")
capitals = {(r.method, r.nu): r.capital for r in golden.rows}
print("suma simple 7:", sino(np.isclose(capitals[("simple-sum", None)], 7.0)), "| fila de raíz cuadrada 6.0828:", sino(np.isclose(capitals[("variance-covariance", None)], 6.0828, atol=1e-4)),
      "| inferior <= raíz cuadrada <= peor:", sino(capitals[("lower-bound", None)] <= capitals[("variance-covariance", None)] <= capitals[("worst-case", None)]))
doc = " ".join(inspect.getdoc(bounds.worst_var_bounds).split())
published = re.search(r"Golden \(Table 4, Pareto\(2\), N = 1e5, epsilon = 1e-3, d = 8\): alpha 0.99 lower ([\d.]+) - ([\d.]+), upper ([\d.]+) - ([\d.]+)", doc).groups()
print("el docstring registra el ejemplo publicado (Tabla 4, Pareto(2), d = 8, alpha 0.99): mejor VaR", published[0], "a", published[1], "| peor VaR", published[2], "a", published[3])

objeto devuelto: Band | capital principal: 7.0 | filas: 7
lower-bound          nu None  capital   0.7601  parte de la suma 0.1086  information-not-capital
simple-sum           nu None  capital   7.0000  parte de la suma 1.0000  headline-capital
variance-covariance  nu None  capital   6.0828  parte de la suma 0.8690  information-not-capital
gaussian-copula      nu None  capital   6.0912  parte de la suma 0.8702  information-not-capital
t-copula             nu 4.0   capital   6.4726  parte de la suma 0.9247  information-not-capital
t-copula             nu 30.0  capital   6.2208  parte de la suma 0.8887  information-not-capital
worst-case           nu None  capital   7.4467  parte de la suma 1.0638  information-not-capital
suma simple 7: sí | fila de raíz cuadrada 6.0828: sí | inferior <= raíz cuadrada <= peor: sí
el docstring registra el ejemplo publicado (Tabla 4, Pareto(2), d = 8, alpha 0.99): mejor VaR 9.00 a 9.00 | peor VaR 141.66 a 141.67


## Estado del arte

Lo que la investigación encontró como la práctica a superar (dossier 05, G.2 y G.3, con el estado de cada afirmación):

- El dossier **prescribe** la banda, "en lugar de una sola cifra diversificada", y advierte contra afirmar que los beneficios de diversificación son robustos; es una recomendación de la investigación, no una descripción de lo que hacen supervisores o herramientas.
- El Joint Forum encontró los modelos de agregación "un trabajo en progreso" en los que los supervisores en general no confían (el comunicado de prensa del 21 de octubre de 2010, https://www.bis.org/press/p101021.htm, trae ambas frases).
- Rosenberg y Schuermann (2006) muestran, en un estudio, que la elección de cópula importa modestamente mientras la aproximación aditiva sobrestima el riesgo en cerca de 30 a 40 por ciento; y Embrechts, Puccetti y Rueschendorf (2013) dan las cotas exactas de peor caso, un método que existe en la literatura.
- La investigación **no encontró una herramienta mantenida ni una práctica publicada que entregue la banda junto a cada agregado** (NO VERIFICADO más allá de la búsqueda del propio dossier).

Lo que existe, entonces, son los ingredientes: las cópulas, las cotas, la referencia y la recomendación. Lo que agrega el caso es **componerlos y no mostrar nunca una cifra sola**, como regla hecha cumplir por contrato (los requisitos CT-1811 y CT-1812 del caso) y por el tipo de retorno del motor.

In [6]:
# Leer las líneas de requisitos del caso que hacen de la banda una regla, para citar sus identificadores y sus puertas
requirements = (ROOT / "docs" / "design" / "features" / "c18" / "requirements.md").read_text(encoding="utf-8").splitlines()
for line in requirements:
    if line.startswith("| CT-1811") or line.startswith("| CT-1812"):
        identifier, text, gate = [cell.strip() for cell in line.strip("|").split("|")[:3]]
        print(identifier, "| la puerta que falla cuando se viola:", gate.split("::")[-1].strip("` "))

CT-1811 | la puerta que falla cuando se viola: test_the_headline_is_the_simple_sum_and_every_other_figure_is_information
CT-1812 | la puerta que falla cuando se viola: test_the_band_has_the_ladder_rows_and_the_euler_allocation_adds_up


## Más allá del estado del arte

**BS-9, la banda de dependencia con cada agregado.** Los pasos anteriores son su evidencia: la banda está en cada agregado versionado, en un orden fijo, con una fila principal y el resto rotulado información; sus filas en forma cerrada se recalculan exactamente; sus reglas estructurales pasan; su motor devuelve la banda completa; y los requisitos CT-1811 y CT-1812 la hacen una regla con puerta del caso. La afirmación se enuncia hasta donde llega la evidencia: **el caso muestra la banda, siempre**. Que la literatura no tenga un equivalente es NO VERIFICADO más allá de la búsqueda de la investigación.

Lo que la banda agrega sobre un solo número es medible en el documento versionado. La celda siguiente informa, para el agregado base, qué habría dicho cada cifra única y qué habría ocultado: el capital principal solo, la cifra gaussiana sola y el peor caso solo, como distancias al resto de la banda.

In [7]:
# Qué habría dicho cada cifra única del agregado base, y la parte de la banda que habría ocultado
rows = {(r["method"], r["nu"]): r["capital_clp_m"] for r in blocks["baseline"]["band"]}
others = lambda key: [v for k, v in rows.items() if k != key]
for label, key in (("el capital principal solo (suma simple)", ("simple-sum", None)), ("la cifra gaussiana sola", ("gaussian-copula", None)), ("el peor caso solo", ("worst-case", None))):
    value = rows[key]
    below, above = sum(v < value for v in others(key)), sum(v > value for v in others(key))
    print(f"{label:40s} {value:10,.0f} | filas de la banda bajo ella: {below}, sobre ella: {above} | rango de las otras como múltiplo de ella: {min(others(key)) / value:.2f} a {max(others(key)) / value:.2f}")
print("la banda como se muestra: de", round(min(rows.values())), "a", round(max(rows.values())), "| el número de filas de la banda sobre el capital principal:", sum(v > rows[("simple-sum", None)] for v in rows.values()))

el capital principal solo (suma simple)     658,046 | filas de la banda bajo ella: 6, sobre ella: 1 | rango de las otras como múltiplo de ella: 0.18 a 1.25
la cifra gaussiana sola                     491,974 | filas de la banda bajo ella: 1, sobre ella: 6 | rango de las otras como múltiplo de ella: 0.24 a 1.67
el peor caso solo                           823,332 | filas de la banda bajo ella: 7, sobre ella: 0 | rango de las otras como múltiplo de ella: 0.14 a 0.80
la banda como se muestra: de 118518 a 823332 | el número de filas de la banda sobre el capital principal: 1


## Resultados

La segunda parte de este cuaderno es una afirmación sobre **lo que no está construido**. Los casos del tema son C18, construido, y C19, la réplica de los modelos supervisores de estrés frente a tasas de pérdida publicadas ("respuesta conocida"), que **sigue planificado**. Este cuaderno no enseña nada de C19. Lee los tres registros que lo dicen: el índice de casos lista C19 con una ruta de manifiesto, el registro de preparación dice que esa ruta falta y da la razón, y el contrato de estrés del motor registra por qué: ningún tipo de cartera de la Reserva Federal es replicable desde el registro público, así que el motor no tiene función de réplica supervisora ni prueba de réplica supervisora. La celda siguiente lee los tres registros y los contrasta entre sí.

In [8]:
# Leer los tres registros que dicen que C19 está planificado y no construido, y contrastarlos entre sí
index = read_json("manifests/index.json")
c19 = next(case for case in index["cases"] if case["case_id"] == "C19")
print("índice:", c19["case_id"], "| tipo:", c19["kind"], "| ruta del manifiesto:", c19["manifest_path"], "| el manifiesto existe:", sino((DERIVED / c19["manifest_path"]).exists()))
readiness = next(contract for contract in read_json("readiness.json")["contracts"] if contract["id"] == "case-manifest")
print("registro de preparación: estado", readiness["state"], "| falta:", readiness["missing"], "| razón:", readiness["reason"]["es"])
contract_text = " ".join((ROOT / "docs" / "contracts" / "engine" / "stress-api.md").read_text(encoding="utf-8").split())
print("el contrato de estrés registra la decisión de mantener C19 planificado:", sino("keeps C19 (supervisory stress model replication) planned" in contract_text))
print("declara que no existe función ni prueba de réplica supervisora:", sino("no supervisory function and no supervisory replication test" in contract_text))
print("C18 construido, C19 planificado: el índice lista", len([case for case in index["cases"] if case["category_id"] == "capital-stress"]), "casos en la categoría de capital y estrés,", "de los cuales", sum((DERIVED / case["manifest_path"]).exists() for case in index["cases"] if case["category_id"] == "capital-stress"), "tiene manifiesto")

índice: C19 | tipo: real | ruta del manifiesto: manifests/C19.json | el manifiesto existe: no
registro de preparación: estado partial | falta: ['manifests/C19.json'] | razón: 1 de 22 casos están planificados y aún no construidos: C19.
el contrato de estrés registra la decisión de mantener C19 planificado: sí
declara que no existe función ni prueba de réplica supervisora: sí
C18 construido, C19 planificado: el índice lista 2 casos en la categoría de capital y estrés, de los cuales 1 tiene manifiesto


- **La afirmación se cumple en cada agregado versionado.** Los cinco documentos llevan seis agregados (incluido el estrés de correlación), pero **solo dos son distintos**: el bloque de agregación de los documentos base, EBA, Fed y NGFS y del documento de cópula t es un mismo bloque (el mismo hash), porque la agregación no depende del escenario, y el sexto es ese bloque con el estrés mercado-crédito 0,8. La evidencia son por lo tanto dos agregados, cada uno con ocho filas de banda en el orden declarado, exactamente una fila rotulada capital principal (la suma simple, CLP 658.046 millones en ambos) y las otras siete rotuladas información, un horizonte y un nivel (un año, 99,9 por ciento), una brecha de Euler entre 0 y $1{,}5 \times 10^{-16}$ y un bloque de riesgo de modelo rotulado información. El peor caso es 1,2512 veces el capital principal en ambos. No se sigue que la banda se haya comprobado bajo cuatro escenarios de estrés: se comprobó sobre el agregado base y sobre el estrés mercado-crédito 0,8.
- **Las filas en forma cerrada se recalculan exactamente en los seis bloques (dos agregados distintos)**, la prueba de intervalo y la de signo (sobre las filas diversificadas) son verdes en los seis, y la fila de raíz cuadrada del documento con estrés, 568.521, está sobre la del base, 528.106.
- **El motor devuelve la banda completa en su ejemplo con valores de referencia**: una `Band` de siete filas (dos grados de libertad t), capital principal 7,0, fila de raíz cuadrada 6,0828 (el valor de referencia), gaussiana 6,0912, t 6,4726 y 6,2208, cota inferior 0,7601 y peor caso 7,4467, que es 1,0638 veces la suma simple. El docstring de `worst_var_bounds` registra el ejemplo publicado (Tabla 4, marginales Pareto(2), $d = 8$) con su rango de peor caso, que el cuaderno 03 reprodujo.
- **Cada cifra única oculta la mayor parte de la banda.** El capital principal solo oculta que el peor caso queda sobre él (6 filas bajo él, 1 sobre él) y que las otras van de 0,18 a 1,25 de él; la cifra gaussiana sola tiene 1 fila bajo ella y 6 sobre ella y las otras van de 0,24 a 1,67 de ella; el peor caso solo tiene las otras siete bajo él, hasta 0,14 de él. La banda abarca un factor de siete, de 118.518 a 823.332.
- **La réplica de los modelos supervisores de estrés, C19, sigue planificada, y tres registros concuerdan.** El índice lista C19 con la ruta `manifests/C19.json`, que no existe; el registro de preparación marca el manifiesto de casos `partial`, nombra esa ruta como faltante y da la razón ("1 de 22 casos están planificados y aún no construidos: C19"); el contrato de estrés del motor registra la decisión de mantener C19 planificado y declara que el motor no tiene función de réplica supervisora ni prueba de réplica supervisora. De los dos casos de la categoría de capital y estrés, uno tiene manifiesto.

Lo que este cuaderno no afirma: que nadie más muestre una banda (NO VERIFICADO más allá de la búsqueda de la investigación); que la banda se derive desde primeros principios aquí (los cuadernos 02 y 03 reconstruyen las marginales desde las grillas de cuantiles versionadas y recuperan las filas versionadas con error menor que 0,01 por ciento; las muestras de un millón de simulaciones detrás de las grillas no están versionadas); ni nada sobre la respuesta de C19, que no está construida.

## Resumen

- BS-9 afirma que toda cifra de capital agregado de C18 se muestra con la banda de dependencia completa, con una fila principal y el resto rotulado información; los documentos versionados la llevan en dos agregados distintos (el base, repetido en cinco documentos, y el estrés mercado-crédito 0,8) y toda verificación de este cuaderno pasa.
- Las filas en forma cerrada de cada banda las recalcula exactamente el motor; las reglas de intervalo y de signo se cumplen; `dependence_band` del motor devuelve una `Band`, nunca un escalar, sobre su propio ejemplo con valores de referencia.
- La afirmación se limita a lo que hace el caso; que nadie más entregue la banda es NO VERIFICADO más allá de la búsqueda de la investigación.
- Una sola cifra oculta la mayor parte de la banda: el capital principal solo oculta que el peor caso queda sobre él, y la banda completa abarca un factor de siete.
- La réplica de los modelos supervisores de estrés, C19, sigue planificada: los tres registros concuerdan, y nada de este tema enseña a partir de ella.

**Qué probar a continuación.** Agrega las cotas exactas a una banda de tus propias marginales con `dependence_band` y mira el ancho de tu propia banda antes de mirar una cifra diversificada.

## Referencias

- Embrechts, P., Puccetti, G. y Rueschendorf, L. (2013), Model uncertainty and VaR aggregation, Journal of Banking and Finance 37(8). https://doi.org/10.1016/j.jbankfin.2013.03.014
- Rosenberg, J. V. y Schuermann, T. (2006), A general approach to integrated risk management with skewed, fat-tailed risks, Journal of Financial Economics 79(3). https://doi.org/10.1016/j.jfineco.2005.03.001
- Joint Forum (2010), Developments in modelling risk aggregation (avances en la modelación de la agregación de riesgos). https://www.bis.org/publ/joint25.pdf
- Banco de Pagos Internacionales (2010), comunicado de prensa del 21 de octubre de 2010 sobre el informe del Joint Forum Developments in modelling risk aggregation. https://www.bis.org/press/p101021.htm
- Dossier 05 de la investigación persistida de Contraste (G.2 y G.3: la evidencia sobre agregación y las trampas de la dependencia), resumido en la base de investigación del SDD.
- Este repositorio: `docs/design/features/c18/requirements.md` (CT-1811 y CT-1812), `docs/contracts/engine/aggregation-api.md` y `docs/contracts/engine/stress-api.md`.